# Week 8-H: Live frozen controller and matched G0/G1 revision

Original SARA generates the shared canonical semantic-k4 draft. Its actual continuation scores feed the frozen B0-GW uncertainty gate. MiniLM draft similarity selects a passage; the actual frozen U4-A model supplies HELP/HURT safety probabilities. All thresholds come from the FIT-selected W8-E configuration. No proxy scorer, validation outcome, or new policy tuning is used.

**G0** blindly regenerates with the promoted passage. **G1** uses that same passage and draft, with revision instructions inside the user chat turn. Actions, passage identities and original drafts are shared, so this isolates the revision prompt. STOP returns the original draft with no additional generation.

With `INCLUDE_W8G=True`, also report all three completed W8-G revision adapters independently: uniform SFT, HC-LoRA and OW-LoRA. No validation winner is selected. QCLA is not stacked onto the base draft because the original gate was not calibrated for that changed distribution. DAFiR-2 is unavailable without state-specific training and sequential outcomes.

Set `RUN_FULL=True` to evaluate 342 INTERNAL-VAL queries after the real GPU smoke passes. This is previously inspected development confirmation; TEST remains unused. Retrieval and SFR embeddings are precomputed benchmark inputs. Timed query-compute components exclude retrieval, cold model loading and deployment overhead; they are not end-to-end deployment wall times.

In [1]:
import sys, subprocess, json
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'configs/week8/w8_dafir.yaml').is_file())
SCRIPT = ROOT / 'scripts/week8/run_w8h_full_evaluation.py'
OUTPUT = ROOT / 'results/week8/w8h_full_system'
RUN_FULL = True  # Set True only in a working GPU-enabled sara_env kernel.
INCLUDE_W8G = True  # Include every completed W8-G revision variant, without choosing a winner.
GPU_AVAILABLE = torch.cuda.is_available()
EXTRA = ['--include-w8g'] if INCLUDE_W8G else []
def run(*args):
    subprocess.run([sys.executable, str(SCRIPT), *map(str,args), *EXTRA], cwd=ROOT, check=True)
print('Python:', sys.executable, '| CUDA available:', GPU_AVAILABLE)
run('--check', '--output-dir', OUTPUT)

Python: /home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/bin/python | CUDA available: True
Inputs verified: 342 queries; live B0-GW + S3/U4-A; paired G0/G1; variants ['original_SARA', 'uniform_SFT', 'HC_LoRA', 'OW_LoRA'] | CUDA available: True | real GPU smoke required before full execution


## Real GPU smoke — required before full evaluation

The smoke covers STOP, PROMOTE and a question-whitespace case where available. It regenerates original drafts, recomputes learned decisions, checks canonical G0 parity, and evaluates matched G0/G1 for the declared revision models. A GPU-unavailable session can inspect CPU verification but cannot establish generation readiness. Missing GPU/data/models raise errors; no fabricated result replaces them.

In [2]:
GPU_SMOKE_PASSED = False
if GPU_AVAILABLE:
    run('--smoke', '--output-dir', OUTPUT)
    smoke = json.loads((OUTPUT / 'smoke/summary.json').read_text())
    assert smoke['live_gate_and_selector_inference'] and smoke['no_validation_selection']
    assert not smoke['TEST_used'] and not smoke['DAFiR2_evaluated']
    GPU_SMOKE_PASSED = True
    display(pd.read_csv(OUTPUT / 'smoke/comparison.csv'))
else:
    print('GPU smoke pending: CUDA is unavailable in this session. Use a working GPU-enabled sara_env kernel/host.')
    cpu = OUTPUT / 'cpu_inference_verification.json'
    if cpu.exists():
        report = json.loads(cpu.read_text())
        print('CPU gate/selector check:', report['action_matches_saved_E5'], '/', report['queries'], 'actions match.')
        print('Maximum U4-A probability difference:', report['U4A_probability_max_abs_diff'])
        print('This checks CPU inference and prompt identities; it is not a GPU generation smoke.')

Verified completed measured W8-H run: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8h_full_system/smoke


,variant,revision_mode,queries,downstream_f1,base_k4_f1,gain_over_k4,exact_match,promotion_rate,hurt_rate_all_queries,help_precision_promotions,hurt_rate_promotions,mean_final_prompt_tokens,mean_cumulative_prompt_tokens,mean_generated_tokens,mean_timed_query_compute_seconds,reference_action_agreement,end_to_end_latency_sec
0,original_SARA,G0,3,0.422222,0.422222,0.000000,0.333333,0.333333,0.000000,0.0,0.0,915.0,1269.333333,12.666667,4.045488,1.0,NaN
1,original_SARA,G1,3,0.422222,0.422222,0.000000,0.333333,0.333333,0.000000,0.0,0.0,936.0,1290.333333,12.666667,4.091033,1.0,NaN
2,uniform_SFT,G0,3,0.422222,0.422222,0.000000,0.333333,0.333333,0.000000,0.0,0.0,915.0,1269.333333,12.666667,4.074040,1.0,NaN
3,uniform_SFT,G1,3,0.088889,0.422222,-0.333333,0.000000,0.333333,0.333333,0.0,1.0,936.0,1290.333333,16.000000,4.273010,1.0,NaN
4,HC_LoRA,G0,3,0.422222,0.422222,0.000000,0.333333,0.333333,0.000000,0.0,0.0,915.0,1269.333333,12.666667,4.077784,1.0,NaN
5,HC_LoRA,G1,3,0.422222,0.422222,0.000000,0.333333,0.333333,0.000000,0.0,0.0,936.0,1290.333333,13.000000,4.137243,1.0,NaN
6,OW_LoRA,G0,3,0.422222,0.422222,0.000000,0.333333,0.333333,0.000000,0.0,0.0,915.0,1269.333333,12.666667,4.084616,1.0,NaN
7,OW_LoRA,G1,3,0.422222,0.422222,0.000000,0.333333,0.333333,0.000000,0.0,0.0,936.0,1290.333333,13.000000,4.147708,1.0,NaN


## Full measured experiment

Base generation and live decisions are resumable per query. Revision generations resume per query/model/mode. All G0/G1 comparisons share the new original-SARA draft and live decision. Historical decision agreement is recorded as a diagnostic; numerical threshold differences are reported rather than overridden. G0 with original SARA must reproduce the corresponding canonical promotion.

Source and artifact hashes guard reuse. A completed run is inspected, not silently retuned. For an explicitly new configuration or changed code, use a fresh output directory. Cold model loading is outside query timings; both generation prompts and all generated tokens are counted. SARA returns continuation tokens only, so prompt length is never subtracted from the output token count.

In [3]:
if RUN_FULL:
    if not GPU_SMOKE_PASSED:
        raise RuntimeError('Full evaluation requires a successful real GPU smoke in the preceding cell.')
    run('--full', '--output-dir', OUTPUT)
    summary = json.loads((OUTPUT / 'full/summary.json').read_text())
    assert summary['live_gate_and_selector_inference'] and not summary['TEST_used']
    display(pd.read_csv(OUTPUT / 'full/comparison.csv'))
else:
    print('Full evaluation has not started. Set RUN_FULL=True on the GPU host and Run All.')

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 15.21it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `m

      variant revision_mode  queries  downstream_f1  base_k4_f1  gain_over_k4  exact_match  promotion_rate  hurt_rate_all_queries  help_precision_promotions  hurt_rate_promotions  mean_final_prompt_tokens  mean_cumulative_prompt_tokens  mean_generated_tokens  mean_timed_query_compute_seconds  reference_action_agreement end_to_end_latency_sec
original_SARA            G0      342       0.400080    0.389061      0.011018     0.236842        0.418129               0.046784                   0.132867              0.111888               1017.017544                    1413.157895              21.052632                          4.355145                    0.997076                   None
original_SARA            G1      342       0.395399    0.389061      0.006338     0.228070        0.418129               0.008772                   0.041958              0.020979               1044.915205                    1441.055556              21.105263                          4.431365                    

,variant,revision_mode,queries,downstream_f1,base_k4_f1,gain_over_k4,exact_match,promotion_rate,hurt_rate_all_queries,help_precision_promotions,hurt_rate_promotions,mean_final_prompt_tokens,mean_cumulative_prompt_tokens,mean_generated_tokens,mean_timed_query_compute_seconds,reference_action_agreement,end_to_end_latency_sec
0,original_SARA,G0,342,0.400080,0.389061,0.011018,0.236842,0.418129,0.046784,0.132867,0.111888,1017.017544,1413.157895,21.052632,4.355145,0.997076,NaN
1,original_SARA,G1,342,0.395399,0.389061,0.006338,0.228070,0.418129,0.008772,0.041958,0.020979,1044.915205,1441.055556,21.105263,4.431365,0.997076,NaN
2,uniform_SFT,G0,342,0.412340,0.389061,0.023279,0.230994,0.418129,0.055556,0.272727,0.132867,1017.017544,1413.157895,21.997076,4.492288,0.997076,NaN
3,uniform_SFT,G1,342,0.396783,0.389061,0.007722,0.230994,0.418129,0.014620,0.062937,0.034965,1044.915205,1441.055556,20.947368,4.505515,0.997076,NaN
4,HC_LoRA,G0,342,0.409215,0.389061,0.020153,0.228070,0.418129,0.067251,0.279720,0.160839,1017.017544,1413.157895,22.698830,4.578758,0.997076,NaN
5,HC_LoRA,G1,342,0.398497,0.389061,0.009436,0.233918,0.418129,0.008772,0.055944,0.020979,1044.915205,1441.055556,21.192982,4.579078,0.997076,NaN
6,OW_LoRA,G0,342,0.403553,0.389061,0.014492,0.225146,0.418129,0.052632,0.244755,0.125874,1017.017544,1413.157895,21.576023,4.518313,0.997076,NaN
7,OW_LoRA,G1,342,0.399715,0.389061,0.010654,0.230994,0.418129,0.011696,0.055944,0.027972,1044.915205,1441.055556,21.213450,4.570846,0.997076,NaN


## Interpretation

Compare G1 versus G0 within each fixed revision model using paired query/paper analyses. Do not switch policies or checkpoints based on these validation results. Report unfavorable results and harmful revisions. Final prompt tokens differ from cumulative prompt tokens; G1's extra draft/instruction tokens are included. The old prototype's utilization labels and proxy-score tables are not valid evidence and are archived separately.

W8-E, W8-F and W8-G result artifacts are retained. Checkpoint-compatible W8-G support source was restored byte-for-byte against its recorded SHA256 hashes. The prior notebook's direct `src.model` import is replaced by a runner that registers the SARA package namespace before loading a model.